# Main-text figures (Figures 2 and 4) and Supplementary S1–S3

Each cell loads a cached result table from `paper/output/` and redraws a figure into `paper/figures/` with the filename used in the manuscript. The tables are produced by `paper/run_main_experiments.py --pop CEU|AFR`; rerun that (needs `magenpy` for CEU) to regenerate them from the genotype data.


In [ ]:
import os, sys
_here = os.getcwd()
for _p in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
    if os.path.isdir(os.path.join(_p, "ld_estimates")):
        REPO = _p; break
else:
    raise RuntimeError("Run this notebook from inside the SCoLD repo.")
sys.path.insert(0, REPO)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
OUT    = os.path.join(REPO, "paper", "output")
FIGDIR = os.path.join(REPO, "paper", "figures")
os.makedirs(FIGDIR, exist_ok=True)
print("repo root:", REPO)


## Figure 2 — bias, RMSE and variance by distance (`Samp`, `Cal`, `mCal`)


In [ ]:
from paper.plotting import plot_rmse_distribution_by_bin, create_estimator_legend
MAIN = {"Samp": "tab:gray", "Cal": "tab:blue", "mCal": "tab:cyan"}
for pop in ["CEU", "AFR"]:
    df = pd.read_csv(os.path.join(OUT, f"metrics_{pop}.csv"))
    df = df[df["method"].isin(MAIN)]
    for stat, name in [("bias", "bias"), ("rmse", "RMSE"), ("variance", "variance")]:
        plot_rmse_distribution_by_bin(df, statistic=stat, colormap=MAIN, show_legend=False,
                                      outpath=os.path.join(FIGDIR, f"{name}_{pop}.pdf"))
        plt.close("all")
create_estimator_legend(MAIN, outpath=os.path.join(FIGDIR, "legend_methods.pdf"))
plt.close("all")
print("wrote bias/RMSE/variance_{CEU,AFR}.pdf and legend_methods.pdf")


## Supplementary S3 — LD-score RMSE


In [ ]:
from paper.plotting import plot_metric_distribution
LD = {"BS": "tab:orange", "Rag": "tab:green", "Supp": "tab:red",
      "Cal": "tab:blue", "mCal": "tab:cyan", "Flex": "tab:purple"}
for pop in ["CEU", "AFR"]:
    ld = pd.read_csv(os.path.join(OUT, f"ldscore_{pop}.csv"))
    cmap = {k: v for k, v in LD.items() if k in set(ld["method"])}
    plot_metric_distribution(ld, statistic="rmse", colormap=cmap,
                             outpath=os.path.join(FIGDIR, f"ld_score_{pop}.png"))
    plt.close("all")
print("wrote ld_score_{CEU,AFR}.png")


## Figure 4 — F1 of LD classification by threshold

*Reconstructed* from the sampled pairs (the original F1 plotting code was not preserved). For each threshold a pair is a true positive of "high LD" when the population value exceeds it and predicted positive when the estimate does.


In [ ]:
import seaborn as sns
F1C = {"Samp": "tab:gray", "BS": "tab:orange", "Rag": "tab:green",
       "Cal": "tab:blue", "mCal": "tab:cyan"}
for pop in ["CEU", "AFR"]:
    fp = os.path.join(OUT, f"f1_{pop}.csv")
    if not os.path.exists(fp):
        print(f"skip {pop}: {os.path.basename(fp)} not found "
              f"(run run_main_experiments.py --pop {pop})"); continue
    f1 = pd.read_csv(fp)
    f1 = f1[f1["method"].isin(F1C)]
    ns = sorted(f1["n"].unique())
    fig, axes = plt.subplots(1, len(ns), figsize=(6 * len(ns), 5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, n in zip(axes, ns):
        sns.boxplot(data=f1[f1["n"] == n], x="thr", y="f1", hue="method",
                    palette=F1C, hue_order=list(F1C), ax=ax)
        ax.set_title(f"{pop}  n={n}"); ax.set_xlabel(r"$r^2$ threshold"); ax.set_ylabel("F1")
        ax.get_legend().remove()
    axes[-1].legend(title="Estimator", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, f"F1_{pop}.pdf"), bbox_inches="tight", dpi=200)
    plt.close(fig)
    print(f"wrote F1_{pop}.pdf")


## Supplementary Tables S1–S2 — metrics averaged over replicates


In [ ]:
for pop in ["CEU", "AFR"]:
    s = pd.read_csv(os.path.join(OUT, f"summary_{pop}.csv"))
    print(f"\n=== {pop}: mean RMSE by (n, distance bin) ===")
    print(s.pivot_table(index=["n", "binlabel"], columns="method", values="rmse").round(3).to_string())
